# 01 — Exploratory Data Analysis
Ames Housing dataset: ~1460 rows, 80 features.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

df = pd.read_csv('../data/train.csv')
print(f'Shape: {df.shape}')
df.head()

## Basic Info

In [ ]:
print('Dtypes:\n')
print(df.dtypes.value_counts())
print(f'\nNumeric cols : {df.select_dtypes(include=np.number).shape[1]}')
print(f'Categorical cols: {df.select_dtypes(include="object").shape[1]}')

## Missing Values Heatmap

In [ ]:
missing = df.isnull().mean().sort_values(ascending=False)
missing = missing[missing > 0]
print(f'{len(missing)} columns have missing values')

fig, ax = plt.subplots(figsize=(10, 6))
missing.plot(kind='bar', ax=ax, color='steelblue')
ax.set_title('Missing Value Rate by Column')
ax.set_ylabel('Fraction missing')
ax.set_xlabel('')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('../data/missing_values.png', bbox_inches='tight')
plt.show()

## SalePrice Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df['SalePrice'], bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('SalePrice — raw')
axes[0].set_xlabel('Sale Price ($)')
axes[0].set_ylabel('Count')

axes[1].hist(np.log1p(df['SalePrice']), bins=50, color='coral', edgecolor='white')
axes[1].set_title('SalePrice — log1p transformed')
axes[1].set_xlabel('log1p(Sale Price)')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.savefig('../data/saleprice_distribution.png', bbox_inches='tight')
plt.show()

print(f'Mean  : ${df["SalePrice"].mean():,.0f}')
print(f'Median: ${df["SalePrice"].median():,.0f}')
print(f'Skew  : {df["SalePrice"].skew():.3f}')

## Correlation Heatmap (top numeric features with SalePrice)

In [ ]:
numeric = df.select_dtypes(include=np.number)
corr    = numeric.corr()['SalePrice'].drop('SalePrice').sort_values(ascending=False)

top20 = corr.head(20)
fig, ax = plt.subplots(figsize=(8, 6))
top20.plot(kind='barh', ax=ax, color='steelblue')
ax.set_title('Top 20 Features Correlated with SalePrice')
ax.set_xlabel('Pearson Correlation')
plt.tight_layout()
plt.savefig('../data/correlations.png', bbox_inches='tight')
plt.show()

print('\nTop 10 positive correlations:')
print(corr.head(10).to_string())

## Correlation Heatmap — numeric feature matrix

In [ ]:
top_cols = corr.head(10).index.tolist() + ['SalePrice']
corr_matrix = df[top_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f',
            cmap='coolwarm', center=0, ax=ax)
ax.set_title('Correlation Heatmap — Top Features')
plt.tight_layout()
plt.savefig('../data/corr_heatmap.png', bbox_inches='tight')
plt.show()

## Categorical Cardinality

In [ ]:
cat_cols = df.select_dtypes(include='object').columns
cardinality = df[cat_cols].nunique().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
cardinality.plot(kind='bar', ax=ax, color='mediumseagreen')
ax.set_title('Categorical Feature Cardinality')
ax.set_ylabel('Unique values')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('../data/cardinality.png', bbox_inches='tight')
plt.show()

print(f'\nTotal categorical features: {len(cat_cols)}')
print(cardinality.to_string())